# Motor Cortex Dynamics — reproduction + interactive viewer

**Stage 1 (descriptive):** PCA + hand-implemented **jPCA** on real M1 data (MC_Maze) → rotational dynamics (Churchland et al. 2012).

**Stage 2 (mechanistic):** **fixed-point analysis** of a task-trained RNN → the local structure that *generates* the rotation (Sussillo & Barak 2013; Sussillo et al. 2015).

jPCA is *descriptive* — one global skew-symmetric linear fit to trajectory geometry, needs no equations, runs on **brain and RNN**. Fixed-point analysis is *mechanistic* — needs the evaluable vector field `dx/dt`, so it runs on the **RNN only**. Fixed points **generate** the rotation jPCA **describes**. The viewer keeps this asymmetry visible: brain = trajectories + jPCA plane; RNN = trajectories + jPCA + fixed points + flow field.

Each section follows: **(1)** markdown math + *why* → **(2)** code → **(3)** inline plotly → **(4)** markdown interpretation + failure modes.

## 0 · Environment setup
Mount Drive first (Colab filesystem is ephemeral), then install deps and import the `python/` modules.

In [ ]:
# --- FIRST CELL: mount Google Drive so downloads/weights persist across sessions ---
# WHY: Colab's local disk is wiped when the runtime recycles. MC_Maze is large and
# RNN training is slow, so cache both on Drive; a fresh session then re-uses them
# instead of re-downloading / re-training.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/neural-dynamics')
    IN_COLAB = True
except ModuleNotFoundError:
    # Running locally (not Colab): fall back to a repo-relative location.
    PROJECT_ROOT = Path.cwd().resolve().parent
    IN_COLAB = False

CACHE_DIR = PROJECT_ROOT / 'cache'   # MC_Maze downloads + trained RNN weights
DATA_DIR  = PROJECT_ROOT / 'data'    # exported web JSON (the 'plate' reads this)
for d in (CACHE_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f'IN_COLAB={IN_COLAB}  CACHE_DIR={CACHE_DIR}  DATA_DIR={DATA_DIR}')

In [ ]:
# --- Dependencies (run once per session) ---
# Hand code is primary; the two git packages are differential-test ORACLES.
# %pip install -q numpy scipy scikit-learn pandas torch h5py nlb_tools dandi plotly ipywidgets nbformat
# %pip install -q git+https://github.com/bantin/jPCA.git
# %pip install -q git+https://github.com/tripdancer0916/pytorch-fixed-point-analysis.git

In [ ]:
# --- Import the pipeline modules (logic lives in python/, not in cells) ---
import sys
REPO = PROJECT_ROOT if (PROJECT_ROOT / 'python').exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO))
# from python import data, pca_jpca, rnn, fixedpoints, flowfield, export  # (enabled as modules fill in)
import plotly.io as pio
pio.renderers.default = 'colab' if IN_COLAB else 'notebook'
print('modules path:', REPO)

## M0 · jPCA hand implementation _(gated — awaiting approval)_
**(1) math/why:** soft-normalize → subtract cross-condition mean per timepoint (Lebedev 2019 critique noted, kept for faithful reproduction) → PCA (k=6) → finite-diff `Ẋ` → fit `Ẋ = M X` with `M = -Mᵀ` (skew-symmetric constrained least squares, closed form) → eigendecompose (±iω) → top plane.
**PASS/FAIL:** fit R² of `Ẋ=MX`; top rotation-plane variance fraction; principal angle vs Antin `jPCA` < 5°, freq within 5%.

In [ ]:
# TODO(M0): jPCA hand implementation + Antin differential test. (Do not start until approved.)

## M1 · Flip-flop calibration (known answer — DO FIRST)
**(1) math/why:** train a small tanh RNN on the 3-bit flip-flop; run the fixed-point finder.
**FALSIFICATION GATE:** exactly **8 stable fixed points** near cube corners, else STOP and debug before touching reaching data. Differential test vs `pytorch-fixed-point-analysis`. Render 8 FPs + trajectories in the minimal viewer.

In [ ]:
# TODO(M1): flip-flop RNN + fixed-point finder + 8-corner gate + minimal viewer.

## M2 · Brain data — PCA + jPCA (Stage 1)
**(1) math/why:** MC_Maze via nlb_tools, 20 ms bins, Gaussian σ=40 ms, soft-norm, aligned on movement onset (prep + movement). Apply the M0 pipeline.
**Adversarial:** does the rotation survive *without* cross-condition-mean subtraction? Report before/after. Brain viewer: trajectories + jPCA plane, **no fixed points**.

In [ ]:
# TODO(M2): MC_Maze load/preprocess + jPCA on brain data + adversarial CCM check + brain viewer.

## M3 · Reaching RNN + jPCA (Stage 2 part 1)
**(1) math/why:** 256-unit continuous-time tanh RNN, `dx/dt = -x + W_rec·φ(x) + W_in·u + b`, `z = W_out·x`; inputs = target/go, output = hand velocity; **metabolic L2-on-rates mandatory**. Trained on the TASK, **not** on spikes.
**Task gate:** report velocity R², require above threshold before any dynamics analysis. Then apply PCA/jPCA to hidden states — does the RNN rotate?

In [ ]:
# TODO(M3): reaching RNN (task-trained) + velocity R² gate + jPCA on hidden states.

## M4 · Reaching RNN fixed points + flow field (Stage 2 part 2)
**(1) math/why:** minimize `q(x)=½‖dx/dt‖²` (Adam, optional L-BFGS polish); ICs sampled **only from states on real trials + noise**; tolerance from the q-**distribution**, not a hard-coded absolute; de-dup by clustering; classify by Jacobian eigenvalues (stable/unstable/saddle/rotational); `dx/dt` on a grid in the top PCA plane.
**Adversarial:** FPs stable to IC re-sampling? Hand vs reference agree after Hungarian alignment? RNN viewer: trajectories + flow + FPs by class.

In [ ]:
# TODO(M4): fixed/slow point finder + Jacobian classification + flow field + RNN viewer + adversarial checks.

## M5 · Unified viewer + deploy
**(1) math/why:** toggle Brain | RNN | side-by-side. The two live in **different spaces** — render each in its own jPCA space, match only visual scale, surface the limitation honestly. Export JSON via `export.py`; deploy the static app to Hugging Face Spaces (Static SDK).

In [ ]:
# TODO(M5): export JSON + unified viewer + deploy.